In [1]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
import os
from langgraph.graph import StateGraph, END, MessagesState
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing_extensions import TypedDict, Literal, Annotated
from langchain.messages import HumanMessage, SystemMessage, AnyMessage
from langgraph.graph.message import add_messages
from langchain.tools import tool
from tavily import TavilyClient
from pydantic import BaseModel, Field
from pprint import pprint
from IPython.display import Markdown, display, Image
from langgraph.prebuilt import ToolNode

In [2]:
load_dotenv(".env")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [3]:
basic_llm = ChatGroq(model="llama-3.1-8b-instant", api_key = GROQ_API_KEY, temperature=0)
advanced_llm = ChatGroq(model="llama-3.1-8b-instant", api_key = GROQ_API_KEY, temperature=0)

In [4]:
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

In [5]:
# Prompts - Read Content from Files
f = open("./prompts/fetching.md")
FETCHING_PROMPT = f.read()
f = open("./prompts/generations.md")
GENERATION_PROMPT = f.read()
f = open("./prompts/optimisation.md")
OPTIMISATION_PROMPT = f.read()
f = open("./prompts/scoring.md")
SCORING_PROMPT = f.read()
f = open("./prompts/validation.md")
VALIDATION_PROMPT = f.read()

FileNotFoundError: [Errno 2] No such file or directory: './prompts/fetching.md'

In [ ]:
# Suppose The Data is:
clean_customer_data = "no data here"

In [ ]:
# Scoring Agent
score_agent = create_agent(
    model=basic_llm,
    tools=[],
    system_prompt=SCORING_PROMPT
)

query = f"""
    Customer data = {clean_customer_data}
"""
score = score_agent.invoke(input={"messages": [{"role": "user", "content": query}]})
clean_score = Markdown(score.get("messages")[-1].content)
clean_score

**Score**: 100 / 100
**Policy Compliance**: NO POLICY
**Areas for Improvement**: None

In [ ]:
# Generation Agent
@tool
def getHowToWriteMarketingOffre(query: str, max_results=3):
    """
        Searches the internet for proven marketing copy techniques and offer structures.

        Args:
            query: Search query for marketing offer strategies.
    """
    response = tavily_client.search(query=query, max_results=max_results)
    results = response.get("results", [])
    content = []
    for res in results:
        content.append({"url": res.get("url", ""), "content": res.get("content", "")})
    return content


def check_policy_rules():
    """
        Validates offer compliance against company promotional guidelines.
    """
    return "the discount should be between 10% and 50%"
policies = check_policy_rules()

offer_agent = create_agent(
    model=basic_llm,
    tools=[getHowToWriteMarketingOffre],
    system_prompt=GENERATION_PROMPT
)

query = f"""
    USER_PROMPT = {client_data} \n\n
    Customer Data = {clean_score} \n\n
    Policies = {policies}
"""
offre = offer_agent.invoke(input={"messages": [{"role": "user", "content": query}]})
offre = Markdown(offre.get("messages")[-1].content)
offre

Based on the customer data and policies, I will create a personalized marketing offer for Aya.

**Marketing Offer:**

Dear Aya,

Thank you for your recent purchase of a t-shirt for $10! We appreciate your business and would like to offer you a special discount on your next purchase.

As a valued customer, we are offering you 20% off your next purchase of any t-shirt. Simply use the code AYA20 at checkout to receive your discount.

We also want to let you know about our current promotion: buy one t-shirt, get 10% off your second t-shirt of equal or lesser value. This is a great opportunity to stock up on your favorite tees and save even more.

Don't forget to follow us on social media to stay up-to-date on the latest designs, promotions, and exclusive offers. We love seeing our customers' style and would be happy to feature you on our page!

Thank you again for your business, and we look forward to serving you again soon.

Best regards,
[Your Company Name]

**Discount Details:**

* 20% off next purchase of any t-shirt
* Code: AYA20
* Valid for one time use only
* Excludes sale items and already discounted products

**Promotion Details:**

* Buy one t-shirt, get 10% off second t-shirt of equal or lesser value
* Valid on all t-shirts in stock
* Excludes sale items and already discounted products

This marketing offer meets the policies of offering a discount between 10% and 50% off, and it is tailored to Aya's recent purchase of a t-shirt. The offer includes a code for a one-time discount, as well as a promotion that encourages Aya to make a second purchase.

In [ ]:
# Validation Agent
validation_agent = create_agent(
    model=basic_llm,
    tools=[],
    system_prompt=SCORING_PROMPT
)

query = f"""
    - Customer Score:\n\n
        {offre} \n\n
    - Policies = {policies}
"""

validation = validation_agent.invoke(input={"messages": [{"role": "user", "content": query}]})
clean_validation = Markdown(validation.get("messages")[-1].content)
clean_validation

**Score**: 50 / 100
**Policy Compliance**: PASSED
**Areas for Improvement**:
- Discount is at the lower end of the allowed range.

In [ ]:
# Optimisation Agent
optimisation_agent = create_agent(
    model=basic_llm,
    tools=[],
    system_prompt=OPTIMISATION_PROMPT
)

query = f"""
    - Optimise this Offre:\n\n
        {offre}
    - Offre Score: \n\n
        {clean_score}
    - Policies: \n\n
        {policies}
"""

new_offre = optimisation_agent.invoke(input={"messages": [{"role": "user", "content": query}]})
clean_new_offre = Markdown(new_offre.get("messages")[-1].content)
clean_new_offre

**Improved Offer**

Get 20% Off Your First Purchase

We're excited to welcome you to our community! As a valued customer, we're offering you an exclusive 20% discount on your first purchase. Use code WELCOME20 at checkout to redeem your discount.

This offer is valid for a limited time only, so don't miss out. Browse our collection of high-quality products and experience the best value for your money.

**Terms and Conditions:**

- The 20% discount is applicable on the first purchase only.
- The discount code WELCOME20 can be used once per customer.
- The offer is valid for a limited time only and can be withdrawn at any time.
- The discount is not applicable on sale items, gift cards, or other promotional offers.

**Pricing:**

- Regular price: $100
- Discounted price: $80 (20% off)

**How to Redeem:**

1. Browse our collection and add your desired products to the cart.
2. Enter the code WELCOME20 at checkout.
3. Click "Apply" to receive your 20% discount.
4. Complete your purchase to enjoy your savings.

Don't miss out on this amazing opportunity to save 20% on your first purchase. Use code WELCOME20 now and experience the best value for your money!

In [ ]:
from typing import TypedDict, List, Literal
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI

llm = basic_llm
# 1. State = What all agents share
class State(TypedDict):
    question: str
    answer: str
    next: Literal["researcher", "writer", "FINISH"]

# 2. Agent 1: Researcher
def researcher(state: State):
    print("🧠 Researcher is searching...")
    result = llm.invoke(f"Give me 3 bullet points about: {state['question']}")
    state["answer"] = result.content
    state["next"] = "writer" # after research, go to writer
    return state

# 3. Agent 2: Writer  
def writer(state: State):
    print("✍️ Writer is writing...")
    result = llm.invoke(f"Turn these bullet points into 1 paragraph: {state['answer']}")
    state["answer"] = result.content
    state["next"] = "FINISH" # we are done
    return state

# 4. Supervisor: The router
def supervisor(state: State):
    print("👔 Supervisor is deciding...")
    if not state["answer"]: # if we have no answer yet
        state["next"] = "researcher"
    else: # if we already have research, go to writer
        state["next"] = "writer"
    
    # If writer already ran, we are done
    if "paragraph" in state["answer"].lower():
        state["next"] = "FINISH"
    return state

# 5. Build Graph
builder = StateGraph(State)

builder.add_node("supervisor", supervisor)
builder.add_node("researcher", researcher) 
builder.add_node("writer", writer)

builder.set_entry_point("supervisor")

# After worker runs, always go back to supervisor
builder.add_edge("researcher", "supervisor")
builder.add_edge("writer", "supervisor")

# Supervisor decides where to go
builder.add_conditional_edges(
    "supervisor",
    lambda x: x["next"],
    {
        "researcher": "researcher",
        "writer": "writer", 
        "FINISH": END
    }
)

graph = builder.compile()

# 6. Run it
final_state = graph.invoke({
    "question": "What is LangGraph?",
    "answer": "",
    "next": "supervisor"
})

print("\n=== FINAL ANSWER ===")
print(final_state["answer"])

👔 Supervisor is deciding...
🧠 Researcher is searching...
👔 Supervisor is deciding...
✍️ Writer is writing...
👔 Supervisor is deciding...
✍️ Writer is writing...
👔 Supervisor is deciding...
✍️ Writer is writing...
👔 Supervisor is deciding...
✍️ Writer is writing...
👔 Supervisor is deciding...
✍️ Writer is writing...
👔 Supervisor is deciding...
✍️ Writer is writing...
👔 Supervisor is deciding...
✍️ Writer is writing...
👔 Supervisor is deciding...
✍️ Writer is writing...
👔 Supervisor is deciding...
✍️ Writer is writing...
👔 Supervisor is deciding...
✍️ Writer is writing...
👔 Supervisor is deciding...
✍️ Writer is writing...
👔 Supervisor is deciding...
✍️ Writer is writing...
👔 Supervisor is deciding...
✍️ Writer is writing...
👔 Supervisor is deciding...
✍️ Writer is writing...
👔 Supervisor is deciding...
✍️ Writer is writing...
👔 Supervisor is deciding...
✍️ Writer is writing...
👔 Supervisor is deciding...
✍️ Writer is writing...
👔 Supervisor is deciding...
✍️ Writer is writing...
👔 Super

KeyboardInterrupt: 